In [2]:
!apt-get update && apt-get install -y libegl1-mesa-dev libgles2-mesa-dev
!pip install pyglet==2.0.7


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,065 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,322 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [12

In [3]:
!pip install smplx trimesh pyrender

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 84.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 735.5/735.5 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.3 MB/s eta 0:00:00
  Created wheel for PyOpenGL: filename=PyOpenGL-3.1.0-py3-none-any.whl size=1745193 sha256=7ada35011deafbb2bba04ec91dbc72c47a8c3a1606152bce75fd0d580f8cd1e8
  Stored in directory: /root/.cache/pip/wheels/5c/d0/77/e69597cdbcb72ea27345036f549a737909bcf17c39789472ce
Successfully built PyOpenGL
  Attempting uninstall: PyOpenGL
    Found existing installation: PyOpenGL 3.1.10
    Uninstalling PyOpenGL-3.1.10:
      Successfully uninstalled PyOpenGL-3.1.10


In [4]:
import torch
torch.cuda.is_available()  # Should return True


True

In [18]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import zipfile
zip_path = "/content/drive/MyDrive/FYP/data/wlsal/word2motion/wlasl_pkls_cropFalse_defult_shape.zip"
extract_path = "/content/drive/MyDrive/FYP/data/wlsal/word2motion/annotations"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete. Files are at:", extract_path)


Extraction complete. Files are at: /content/drive/MyDrive/FYP/data/wlsal/word2motion/annotations


In [5]:
#!/usr/bin/env python3
"""
Robust SignAvatars Animation Generator with EGL Offscreen Rendering
====================================================================
This script diagnoses PKL files and renders animations headlessly using EGL.
"""

import os
# Enable EGL offscreen before any OpenGL imports
os.environ['PYOPENGL_PLATFORM'] = 'egl'

import json
import pickle
import numpy as np
import torch
import cv2
from pathlib import Path
import trimesh
import pyrender
import traceback

try:
    import smplx
    SMPLX_AVAILABLE = True
except ImportError:
    SMPLX_AVAILABLE = False


class RobustSignLanguageAnimator:
    def __init__(self, dataset_path, model_path):
        self.dataset_path = Path(dataset_path)
        self.model_path = Path(model_path)
        self.word2motion_path = self.dataset_path / "word2motion"
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")

        raw = self.load_word_mappings_raw()
        self.word_mappings = self.filter_existing_files(raw)
        print(f"Found {len(self.word_mappings)} valid words")

        self.setup_smplx_model()

    def load_word_mappings_raw(self):
        jp = self.dataset_path / "word2motion" / "text" / "WLASL_v0.3.json"
        data = json.load(open(jp, 'r'))
        m = {}
        for e in data:
            w = e['gloss'].lower()
            for inst in e.get('instances', []):
                m[w] = f"{inst['video_id']}.pkl"
        return m

    def filter_existing_files(self, raw):
        ann = self.word2motion_path / "annotations" / "wlasl_pkls_cropFalse_defult_shape"
        valid = {}
        for w, fn in raw.items():
            p = ann / fn
            if p.exists() and p.stat().st_size > 1000:
                valid[w] = fn
        return valid

    def setup_smplx_model(self):
        if not SMPLX_AVAILABLE:
            print("SMPL-X unavailable")
            return
        try:
            self.smplx_model = smplx.create(
                model_path=str(self.model_path),
                model_type='smplx',
                gender='neutral',
                use_face_contour=False,
                flat_hand_mean=True,
                use_pca=True,
                num_pca_comps=45
            ).to(self.device)
            print("SMPL-X model loaded")
        except Exception as e:
            print("Failed SMPL-X load:", e)
            self.smplx_model = None

    def load_motion_data(self, word):
        fn = self.word_mappings.get(word)
        if not fn:
            return None
        p = self.word2motion_path / "annotations" / "wlasl_pkls_cropFalse_defult_shape" / fn
        return pickle.load(open(p, 'rb'))

    def extract_smplx_parameters(self, data):
        allp = data['smplx']
        arr = allp.cpu().numpy() if torch.is_tensor(allp) else np.array(allp)
        params = {
            'root_pose': torch.tensor(arr[:, :3]),
            'body_pose': torch.tensor(arr[:, 3:66]),
            'left_hand_pose': torch.tensor(arr[:, 66:111]),
            'right_hand_pose': torch.tensor(arr[:, 111:156]),
            'jaw_pose': torch.tensor(arr[:, 156:159]),
            'shape': torch.tensor(arr[:, 159:169]),
            'expression': torch.tensor(arr[:, 169:179]),
            'cam_trans': torch.tensor(arr[:, 179:182]),
            'transl': torch.zeros((arr.shape[0], 3))
        }
        return params

    def generate_mesh_sequence(self, params):
        if not SMPLX_AVAILABLE or self.smplx_model is None:
            return []
        meshes = []
        with torch.no_grad():
            for i in range(params['root_pose'].shape[0]):
                inp = {k: v[i:i+1].to(self.device) for k, v in params.items()}
                out = self.smplx_model(**inp)
                verts = out.vertices[0].cpu().numpy()
                meshes.append(trimesh.Trimesh(vertices=verts, faces=self.smplx_model.faces))
        return meshes

    def render_animation(self, meshes, output_path, fps=24):
        scene = pyrender.Scene()
        scene.add(pyrender.DirectionalLight(color=np.ones(3), intensity=3.0))
        cam = pyrender.PerspectiveCamera(yfov=np.pi/3.0)
        scene.add(cam, pose=np.array([[1,0,0,0],[0,1,0,0],[0,0,1,3],[0,0,0,1]]))
        # EGL offscreen renderer
        renderer = pyrender.OffscreenRenderer(800, 600)
        writer = cv2.VideoWriter(output_path,
                                 cv2.VideoWriter_fourcc(*'mp4v'),
                                 fps, (800, 600))
        for mesh in meshes:
            node = scene.add(pyrender.Mesh.from_trimesh(mesh))
            color, _ = renderer.render(scene)
            frame = cv2.cvtColor(color, cv2.COLOR_RGB2BGR)
            writer.write(frame)
            scene.remove_node(node)
        writer.release()
        renderer.delete()
        print("Saved:", output_path)

    def text_to_animation(self, text, out="sign.mp4"):
        w = text.strip().lower().split()[0]
        data = self.load_motion_data(w)
        if data is None:
            print("No data for", w)
            return False
        params = self.extract_smplx_parameters(data)
        meshes = self.generate_mesh_sequence(params)
        if not meshes:
            print("No meshes generated")
            return False
        self.render_animation(meshes, out)
        return True


if __name__ == "__main__":
    DATASET = "/content/drive/MyDrive/FYP/data/wlsal"
    MODEL    = "/content/drive/MyDrive/FYP/models"

    animator = RobustSignLanguageAnimator(DATASET, MODEL)
    available = list(animator.word_mappings.keys())
    print("Available words:", available[:20])
    if available:
        animator.text_to_animation(available[0], f"anim_{available[0]}.mp4")


Using device: cuda
Found 112 valid words
SMPL-X model loaded
Available words: ['all', 'accident', 'apple', 'africa', 'about', 'approve', 'arrive', 'always', 'animal', 'argue', 'afternoon', 'age', 'alone', 'appointment', 'australia', 'adult', 'after', 'ago', 'allow', 'america']
Saved: anim_all.mp4


In [6]:
from IPython.display import Video

# Path to your generated file
video_path = "anim_all.mp4"  # or whatever filename you used

# Display the video inline
Video(video_path, embed=True, width=640, height=480)
